# Model to transform the gas to an hydrogen grid

### Import packages

In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import networkx as nx
import os

### Import Data

In [2]:
# Specify the path to your Excel file
input_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
#excel_file_path = '\Data_update.xlsx'
excel_file_path = '\Data_update.xlsx'

input_file_path  = input_file_path + excel_file_path
full_input_path = os.path.abspath(os.path.join(os.getcwd(), input_file_path))


# Read the Excel file into a DataFrame
df_nodes = pd.read_excel(full_input_path, sheet_name='Nodes')
df_commodities = pd.read_excel(full_input_path, sheet_name='Commodities')
df_edges = pd.read_excel(full_input_path, sheet_name='Edges')
df_parameter = pd.read_excel(full_input_path, sheet_name='Parameters')
df_supply_values = pd.read_excel(full_input_path, sheet_name='Supply')

### Create input data structure

In [3]:
# Extract nodes, edges and commodities from the DataFrames
Network_nodes = df_nodes['Nodes'].dropna().tolist()
Commodities = df_commodities['Commodities'].dropna().tolist()
Edges = list(zip(df_edges['Source'], df_edges['Destination']))

# Create a nested dictionary for initial capacities
Initial_capacities = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    initial_capacity = row['initial_capacities']

    edge = f"{source}{destination}"

    if commodity not in Initial_capacities:
        Initial_capacities[commodity] = {}

    Initial_capacities[commodity][edge] = initial_capacity

# Create a nested dictionary for max capacities
Max_capacities = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    max_capacity = row['max_capacities']

    edge = f"{source}{destination}"

    if commodity not in Max_capacities:
        Max_capacities[commodity] = {}

    Max_capacities[commodity][edge] = max_capacity

# Create a nested dictionary for edge cost
Edge_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    edge_cost = row['costs_edge']

    edge = f"{source}{destination}"

    if commodity not in Edge_cost:
        Edge_cost[commodity] = {}

    Edge_cost[commodity][edge] = edge_cost

# Create a nested dictionary for new pipelines
Pipe_new_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    new_cost = row['new_build_cost']

    edge = f"{source}{destination}"

    if commodity not in Pipe_new_cost:
        Pipe_new_cost[commodity] = {}

    Pipe_new_cost[commodity][edge] = new_cost

# Create a nested dictionary for pipeline conversion
Pipe_conv_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    conv_cost = row['conversion_cost']

    edge = f"{source}{destination}"

    if commodity not in Pipe_conv_cost:
        Pipe_conv_cost[commodity] = {}

    Pipe_conv_cost[commodity][edge] = conv_cost

# Create a nested dictionary for adjusting the capacity when pipeline conversion
Pipe_conv_factor = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    conv_cap_factor = row['conversion_capacity_factor']

    edge = f"{source}{destination}"

    if commodity not in Pipe_conv_factor:
        Pipe_conv_factor[commodity] = {}

    Pipe_conv_factor[commodity][edge] = conv_cap_factor

# Create a nested dictionary for supply values, skipping 0 and NaN values
Supply_values = {}
for index, row in df_supply_values.iterrows():
    commodity = row['Commodity']
    supply_node = row['Node']
    supply_value = row['Supply']

    if commodity not in Supply_values:
        Supply_values[commodity] = {}

    # Skip 0 and NaN values
    if not pd.isna(supply_value) and supply_value != 0:
        Supply_values[commodity][supply_node] = supply_value

# Create a nested dictionary for node values, skipping 0 and NaN values
Node_values = {}
for index, row in df_supply_values.iterrows():
    commodity = row['Commodity']
    demand_node = row['Node']
    node_value = row['Supply']

    if commodity not in Node_values:
        Node_values[commodity] = {}

    # Skip 0 and NaN values
    #if not pd.isna(node_value) and node_value != 0:
    if not pd.isna(node_value):
        Node_values[commodity][demand_node] = node_value

In [4]:
'''
Create slack nodes for all supply nodes
link a node to supply in case of shortage in the system to all supply nodes
the capacity is infinite but at infinite (super high) cost
'''

# Create a new dictionary for positive values
positive_values_dict = {}

# Iterate through the outer dictionary
for node, values in Node_values.items():
    # Filter out positive values from the inner dictionary
    positive_values = {key: value for key, value in values.items() if value > 0}
    
    # Check if there are positive values before adding to the new dictionary
    if positive_values:
        positive_values_dict[node] = positive_values

print("Dictionary with positive values:", positive_values_dict)

# Get all keys from the inner dictionaries
all_keys = [key for values in positive_values_dict.values() for key in values.keys()]

# Remove duplicates to get unique keys
unique_keys = list(set(all_keys))

print("Unique keys:", unique_keys)

# Create a list with names "shortage_" followed by each key
shortage_list = [f'shortage_{key}' for key in unique_keys]

print("Shortage list:", shortage_list)

#shortage_edges_list = [(key, shortage) for key, shortage in zip(unique_keys, shortage_list)]
shortage_edges_list = [(key, shortage) for key, shortage in zip(shortage_list, unique_keys)]


print("Edges list:", shortage_edges_list)

shoratege_capacity_dict = {commodity: {f'{key}{shortage}': 1000 for key, shortage in zip(shortage_list, unique_keys)} for commodity in Commodities}
shoratege_max_capacity_dict = shoratege_capacity_dict
print("Nested dictionary with capacities:", shoratege_capacity_dict)

shoratege_cost_dict = {commodity: {f'{key}{shortage}': 10000000 for key, shortage in zip(shortage_list, unique_keys)} for commodity in Commodities}
print("Shortage cost:", shoratege_cost_dict)


Dictionary with positive values: {'Methane': {'S1': 5, 'S2': 76, 'S3': 15}, 'Hydrogen': {'S1': 20, 'S2': 81, 'S3': 15}}
Unique keys: ['S1', 'S3', 'S2']
Shortage list: ['shortage_S1', 'shortage_S3', 'shortage_S2']
Edges list: [('shortage_S1', 'S1'), ('shortage_S3', 'S3'), ('shortage_S2', 'S2')]
Nested dictionary with capacities: {'Methane': {'shortage_S1S1': 1000, 'shortage_S3S3': 1000, 'shortage_S2S2': 1000}, 'Hydrogen': {'shortage_S1S1': 1000, 'shortage_S3S3': 1000, 'shortage_S2S2': 1000}}
Shortage cost: {'Methane': {'shortage_S1S1': 10000000, 'shortage_S3S3': 10000000, 'shortage_S2S2': 10000000}, 'Hydrogen': {'shortage_S1S1': 10000000, 'shortage_S3S3': 10000000, 'shortage_S2S2': 10000000}}


Print data structure for control

In [5]:
# Print the data for control
#print("Network Nodes:", Network_nodes)
#print("Commodities:", Commodities)
#print("Edges:", Edges)
#print("Initial Capacities:", Initial_capacities)
#print("Max Capacities:", Max_capacities)
#print("Costs per edge Capacities:", Edge_cost)
#print("Costs for new pipelines:", Pipe_new_cost)
#print("Costs for convert pipelines:", Pipe_conv_cost)
#print("Conversion capacity factor:", Pipe_conv_factor)
#print("Node Values:", Node_values)

In [6]:
#check in all nodes are connected via edges

G = nx.Graph()
G.add_edges_from(Edges)

# Check if the graph is connected
if nx.is_connected(G):
    print("The graph is connected.")
else:
    print("The graph is not connected.")

The graph is connected.


In [7]:
#check in all nodes are connected via edges

G = nx.Graph()
G.add_edges_from(Edges)

# Get connected components
connected_components = list(nx.connected_components(G))

# Check if the graph is connected
if len(connected_components) == 1:
    print("The graph is connected.")
else:
    print("The graph is not connected.")
    print("Connected components:")
    for i, component in enumerate(connected_components):
        print(f"Component {i+1}: {component}")

The graph is connected.


In [8]:
# Check if all nodes are connected
if set(Network_nodes) in nx.connected_components(G):
    print("All nodes are connected.")
else:
    print("Not all nodes are connected.")

All nodes are connected.


In [9]:
#implement factor to adjust capacity when conversion from methane to hydrogen
#TODO Implement it from the input file and use a correct factor
conversion_factor = Pipe_conv_factor

## Model

### Create model

In [10]:
# Create a new model
model = gp.Model("Grid_Transformation")

Set parameter Username
Academic license - for non-commercial use only - expires 2024-12-20


### Define parameters

In [11]:
# Parameters
commodities = Commodities  # Commodity types
#real network elements
network_nodes = Network_nodes # Nodes of the system
network_edges = Edges  # Edges
initial_capacities = Initial_capacities # Initial capacities
max_capacities = Max_capacities  # Maximum capacities
costs_edge = Edge_cost  # Cost to transport from node to node
capacity_new_cost = Pipe_new_cost  # Cost to increase capacity
capacity_change_cost = Pipe_conv_cost  # Cost to increase capacity
node_value = Node_values #contains supply and demand values

#slack parameters
slack_nodes = shortage_list
slack_edges = shortage_edges_list
slack_capacities = shoratege_capacity_dict
slack_cost = shoratege_cost_dict

#complete network of the model
all_edges = network_edges + shortage_edges_list

### Define decision variables

In [12]:
# Decision variables
x_flow = {} #flow of commodity on an edge
x_flow_shortage = {}
y_new_cap = {} #new build capacity for a commodity on an edge between two edges
z_conv_cap = {} #capacity of a commodity converted on an edge between two nodes
Change = {} # Binary variable for switching

for commodity in commodities:
    x_flow[commodity] = {}
    x_flow_shortage[commodity] = {}
    y_new_cap[commodity] = {}
    z_conv_cap[commodity] = {}
    Change[commodity] = {}
    for edge in all_edges:
        x_flow[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_{commodity}_{edge[0]}_{edge[1]}")
        y_new_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"y_new_cap{commodity}_{edge[0]}_{edge[1]}")
        z_conv_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"z_conv_cap{commodity}_{edge[0]}_{edge[1]}")
        Change[commodity][edge] = model.addVar(vtype=GRB.BINARY, name=f"change_{commodity}_{edge[0]}_{edge[1]}")
    for edge in slack_edges:
        x_flow_shortage[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_shortage_{commodity}_{edge[0]}_{edge[1]}")
model.update()

### Define objective and constraints

In [13]:
# Objective function (minimize total transportation cost + cost to increase and convert capacity)
model.setObjective(
    gp.quicksum(x_flow[commodity][edge] * costs_edge[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in network_edges) +
    gp.quicksum(y_new_cap[commodity][edge] * capacity_new_cost[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in network_edges) +
    gp.quicksum(z_conv_cap[commodity][edge] * capacity_change_cost[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in network_edges)+
    gp.quicksum(x_flow_shortage[commodity][edge] * slack_cost[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in slack_edges),
    GRB.MINIMIZE
)

# Constraints

#inflow of a node must equal the outflow of a node
for node in network_nodes:  
    for commodity in commodities:  
        # Flow conservation constraint for the current node and commodity
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    - gp.quicksum(x_flow[commodity][edge] for edge in network_edges if edge[0] == node)
                                    + node_value[commodity][node] 
                                    == 0, f"flow_constraint_{commodity}_{node}")
        #model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node or edge[0] == node) 
        #                        - gp.quicksum(x_flow[commodity][edge] for edge in network_edges if edge[0] == node or edge[1] == node)
        #                        + node_value[commodity][node] 
        #                        == 0, f"flow_constraint_{commodity}_{node}")


#slack node at the sources to avoid infeasible problems.        
for node in slack_nodes:
    for commodity in commodities: 
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[0] == node) >= 0,
                        f"supply_shortage_{commodity}_{node}")


# Add constraint for equality between x_flow and x_flow_shortage for the same edge
for commodity in commodities:
    for edge in slack_edges:
        # Ensure equality for the corresponding edges
        model.addConstr(x_flow[commodity][edge] == x_flow_shortage[commodity][(edge[0], edge[1])], 
                        f"equality_constraint_{commodity}_{edge}")


#Capacity constraint for flow
for commodity in commodities:
    for edge in network_edges:
        model.addConstr(x_flow[commodity][edge] 
                        <= y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] #+ initial_capacities[commodity][f"{edge[0]}{edge[1]}"]
                        , f"used_capacity_{commodity}_{edge[0]}_{edge[1]}")


# Capacity constraint for maximal capacity
for commodity in commodities:
    for edge in network_edges:
        model.addConstr(y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] 
                        <= max_capacities[commodity][f"{edge[0]}{edge[1]}"], f"capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraints for edge conversion
for edge in network_edges:
    model.addConstr(Change[Commodities[0]][edge] + Change[Commodities[1]][edge] == 1, f"switching_constraint_{edge[0]}_{edge[1]}")

for commodity in commodities:
    for edge in network_edges:
        model.addConstr(initial_capacities[Commodities[0]][f"{edge[0]}{edge[1]}"] * Change[commodity][edge] #* conversion_factor[commodity][node]
                        == z_conv_cap[commodity][edge], f"changed_capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraing no negative flow
for commodity in commodities:
    for edge in all_edges:
        model.addConstr(x_flow[commodity][edge] >= 0, f"non_negativity_x_{commodity}_{edge[0]}_{edge[1]}")

### Optimize the model

In [14]:
# Optimize the model
model.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 170 rows, 142 columns and 338 nonzeros
Model fingerprint: 0x95dabe3d
Variable types: 108 continuous, 34 integer (34 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+02]
  Objective range  [5e-01, 1e+07]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+02]
Found heuristic solution: objective 1.500070e+08
Presolve removed 169 rows and 139 columns
Presolve time: 0.01s
Presolved: 1 rows, 3 columns, 3 nonzeros
Found heuristic solution: objective 1.500053e+08
Variable types: 0 continuous, 3 integer (1 binary)

Explored 0 nodes (0 simplex iterations) in 0.06 seconds (0.00 work units)
Thread count was 8 (of 8 available processors)

Solution count 2: 1.50005e+08 1.50007e+08 

Optimal solution found (toler

### Results processing

In [15]:
# Print the results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found!")
    for commodity in commodities:
        for edge in network_edges:
            print(f"{commodity}, {edge}: "
                  f"Flow of Commodity = {x_flow[commodity][edge].x}, "
                  f"New Capacity = {y_new_cap[commodity][edge].x}, "
                  f"Switched = {Change[commodity][edge].x}, "
                  f"Changed Capacity = {z_conv_cap[commodity][edge].x}, ")
        for edge in slack_edges:    
           print(f"{commodity}, {edge}: "
                 f"Flow of Shortage = {x_flow_shortage[commodity][edge].x}")
    print("****************************")
    print(f"Total cost: {model.objVal}")
else:
    print("No optimal solution found.")

Optimal solution found!
Methane, ('S1', 'D1'): Flow of Commodity = 0.0, New Capacity = 0.0, Switched = 1.0, Changed Capacity = 500.0, 
Methane, ('S1', 'D2'): Flow of Commodity = 5.0, New Capacity = 0.0, Switched = 1.0, Changed Capacity = 500.0, 
Methane, ('S2', 'D2'): Flow of Commodity = 15.0, New Capacity = 0.0, Switched = 1.0, Changed Capacity = 500.0, 
Methane, ('S2', 'D3'): Flow of Commodity = 76.0, New Capacity = 76.0, Switched = 0.0, Changed Capacity = 0.0, 
Methane, ('S3', 'D1'): Flow of Commodity = 15.0, New Capacity = 0.0, Switched = 1.0, Changed Capacity = 500.0, 
Methane, ('D1', 'D2'): Flow of Commodity = 0.0, New Capacity = 0.0, Switched = 1.0, Changed Capacity = 500.0, 
Methane, ('D3', 'D4'): Flow of Commodity = 51.0, New Capacity = 51.0, Switched = 0.0, Changed Capacity = 0.0, 
Methane, ('D4', 'D5'): Flow of Commodity = 40.0, New Capacity = 40.0, Switched = 0.0, Changed Capacity = 0.0, 
Methane, ('D4', 'D6'): Flow of Commodity = 10.0, New Capacity = 0.0, Switched = 1.0, C

In [16]:
# Assuming commodities, edges, x_flow, y_new_cap, Change, and z_conv_cap are defined in your code

# Create lists to store the data
results_data = []
columns = ["Commodity", "Edge", "Flow", "New Capacity", "Switched", "Changed Capacity"]
shortage_data = []
shortage_columns = ["Commodity", "Edge", "Flow"]

# Check if the model has an optimal solution
if model.status == GRB.OPTIMAL:
    for commodity in commodities:
        for edge in network_edges:
            # Append data to the list
            results_data.append([commodity,
                                 edge, 
                                 x_flow[commodity][edge].x, 
                                 y_new_cap[commodity][edge].x, 
                                 Change[commodity][edge].x, 
                                 z_conv_cap[commodity][edge].x])
        for edge in slack_edges:    
            shortage_data.append([commodity, 
                              edge, 
                              x_flow_shortage[commodity][edge].x])
else:
    print("No optimal solution found.")

if model.status == GRB.OPTIMAL:
    # Create a DataFrame
    results_df = pd.DataFrame(results_data, columns=columns)
    shortage_df = pd.DataFrame(shortage_data, columns=shortage_columns)
    # Print the DataFrame
    print(results_df)
    print('********************************')
    print(shortage_df)

   Commodity       Edge  Flow  New Capacity  Switched  Changed Capacity
0    Methane   (S1, D1)   0.0           0.0       1.0             500.0
1    Methane   (S1, D2)   5.0           0.0       1.0             500.0
2    Methane   (S2, D2)  15.0           0.0       1.0             500.0
3    Methane   (S2, D3)  76.0          76.0       0.0               0.0
4    Methane   (S3, D1)  15.0           0.0       1.0             500.0
5    Methane   (D1, D2)   0.0           0.0       1.0             500.0
6    Methane   (D3, D4)  51.0          51.0       0.0               0.0
7    Methane   (D4, D5)  40.0          40.0       0.0               0.0
8    Methane   (D4, D6)  10.0           0.0       1.0             500.0
9    Methane   (D6, D7)  10.0           0.0       1.0             500.0
10   Methane   (D5, D8)  10.0           0.0       1.0             500.0
11   Methane   (D5, D9)  10.0           0.0       1.0             500.0
12   Methane  (D9, D10)   0.0           0.0       1.0           

In [17]:
#model.write()

In [18]:
results_df

,Commodity,Edge,Flow,New Capacity,Switched,Changed Capacity
0,Methane,"(S1, D1)",0.0,0.0,1.0,500.0
1,Methane,"(S1, D2)",5.0,0.0,1.0,500.0
2,Methane,"(S2, D2)",15.0,0.0,1.0,500.0
3,Methane,"(S2, D3)",76.0,76.0,0.0,0.0
4,Methane,"(S3, D1)",15.0,0.0,1.0,500.0
5,Methane,"(D1, D2)",0.0,0.0,1.0,500.0
6,Methane,"(D3, D4)",51.0,51.0,0.0,0.0
7,Methane,"(D4, D5)",40.0,40.0,0.0,0.0
8,Methane,"(D4, D6)",10.0,0.0,1.0,500.0
9,Methane,"(D6, D7)",10.0,0.0,1.0,500.0


In [19]:
search_element = 'DE'

# Filter rows based on whether the search element is present in 'Column1'
string_df = results_df[results_df['Edge'].astype(str).str.contains(search_element, case=False, na=False)]

# Display the resulting DataFrame
string_df

,Commodity,Edge,Flow,New Capacity,Switched,Changed Capacity


In [20]:
shortage_df

,Commodity,Edge,Flow
0,Methane,"(shortage_S1, S1)",0.0
1,Methane,"(shortage_S3, S3)",0.0
2,Methane,"(shortage_S2, S2)",15.0
3,Hydrogen,"(shortage_S1, S1)",0.0
4,Hydrogen,"(shortage_S3, S3)",0.0
5,Hydrogen,"(shortage_S2, S2)",0.0
